# **Introduction to Deep Learning**
# **Recitation 0.4 – PyTorch (Updated)**

> **What you will build today:** A fully working MNIST digit classifier from scratch — covering tensors, devices, autograd, loss functions, optimizers, and the complete training loop.
>
> **New sections added:** `requires_grad` internals, computational graphs, `model.train()` vs `model.eval()`, `torch.no_grad()`, and a beginner pitfalls checklist.

---
**References:**
- [PyTorch Docs](https://docs.pytorch.org/docs/stable/index.html)
- [PyTorch Beginner Tutorials](https://docs.pytorch.org/tutorials/beginner/basics/intro.html)
- [torch.nn Reference](https://docs.pytorch.org/docs/stable/nn.html)
- [torch.optim Reference](https://docs.pytorch.org/docs/stable/optim.html)

## **1. Installing & Importing PyTorch**

For more install options (CUDA version, conda, etc.): https://pytorch.org/get-started/locally/

In [ ]:
!pip install torch torchvision --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

---
## **2. Tensors — The Building Block**

A **tensor** is an n-dimensional array — the fundamental data structure in PyTorch.

| Dimensions | Name | Example |
|---|---|---|
| 0D | Scalar | `torch.tensor(3.14)` |
| 1D | Vector | `torch.tensor([1, 2, 3])` |
| 2D | Matrix | `torch.zeros(3, 4)` |
| 3D+ | Tensor | Images, batches, sequences |

In [ ]:
# Creating tensors from data
x_list   = [[1, 3, 5], [2, 4, 6]]
x_tensor = torch.tensor(x_list)

print("Tensor:\n", x_tensor)
print("Shape   :", x_tensor.shape)   # torch.Size([2, 3])
print("Ndim    :", x_tensor.ndim)    # 2
print("Dtype   :", x_tensor.dtype)   # torch.int64
print("Device  :", x_tensor.device)  # cpu

In [ ]:
# Common constructors
zeros  = torch.zeros(2, 3)               # all 0s
ones   = torch.ones(3, 3)                # all 1s
full   = torch.full((4, 3), fill_value=6.5)  # constant
rand   = torch.randn(2, 3)              # standard normal
arange = torch.arange(0, 10, step=2)    # [0, 2, 4, 6, 8]

print("zeros :", zeros)
print("rand  :", rand)
print("arange:", arange)

In [ ]:
# NumPy ↔ PyTorch (they share memory on CPU!)
np_arr = np.array([[1, 2], [3, 4]])
t      = torch.from_numpy(np_arr)   # NumPy → Tensor
back   = t.numpy()                  # Tensor → NumPy (CPU only)

print("From numpy:", t)
print("Back to numpy:", back)

# ⚠️ They share memory — modifying one changes the other!
np_arr[0, 0] = 999
print("Tensor after modifying np_arr:", t)   # also changed!

---
## **3. Devices — CPU vs GPU**

PyTorch can run on CPU or GPU (CUDA). GPUs are massively parallel and can speed up training **10–100×** for large networks.

> **Rule:** Your model and all its input tensors must be on the **same device**. Mixing devices causes a `RuntimeError`.

In [ ]:
# Standard device setup — always write it this way
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

x = torch.randn(2, 3)
print("Before .to():", x.device)   # cpu

x = x.to(device)
print("After  .to():", x.device)   # cuda:0 (or still cpu)

In [ ]:
# ❌ This would crash if model is on GPU but inputs are on CPU:
# pred = model(inputs)  →  RuntimeError: Expected all tensors on same device

# ✅ Always move your batch inside the training loop:
# inputs, labels = inputs.to(device), labels.to(device)

---
## **4. Vectorization — Think in Tensors, Not Loops**

PyTorch operations run in optimized C++/CUDA. A vectorized operation is almost always **faster** than a Python loop — and more readable.

In [ ]:
x = torch.tensor([1, 2, 3, 4, 5], dtype=torch.float32)

# ❌ Non-vectorized (slow Python loop)
result_loop = torch.zeros_like(x)
for i in range(len(x)):
    result_loop[i] = x[i] + 5

# ✅ Vectorized (fast, readable)
result_vec = x + 5

print("Loop:", result_loop)
print("Vec :", result_vec)
print("Same?", torch.allclose(result_loop, result_vec))

### 4.1 Dot Product

$$\mathbf{x} \cdot \mathbf{y} = \sum_{i=1}^n x_i y_i$$

**Your Task:** Implement `PYTORCH_dot` using a single PyTorch function.

In [ ]:
def PYTORCH_dot(x, y):
    """
    Dot product of two 1D tensors.
    Hint: torch.dot  OR  (x * y).sum()
    """
    return NotImplemented   # TODO

# Test
np.random.seed(0)
X = torch.from_numpy(np.random.randint(-1000, 1000, size=3000))
Y = torch.from_numpy(np.random.randint(-1000, 1000, size=3000))
print(PYTORCH_dot(X, Y))   # Expected: 7082791

### 4.2 Outer Product

$$[\mathbf{x} \otimes \mathbf{y}]_{ij} = x_i y_j$$

**Your Task:** Implement `PYTORCH_outer`.

In [ ]:
def PYTORCH_outer(x, y):
    """Hint: torch.outer  OR  x.unsqueeze(1) * y.unsqueeze(0)"""
    return NotImplemented   # TODO

np.random.seed(0)
X = torch.from_numpy(np.random.randint(-1000, 1000, size=3000))
Y = torch.from_numpy(np.random.randint(-1000, 1000, size=3000))
print(PYTORCH_outer(X, Y))   # Expected shape: [3000, 3000]

### 4.3 Hadamard (Element-wise) Product

$$[\mathbf{x} \odot \mathbf{y}]_i = x_i y_i$$

**Your Task:** Implement `PYTORCH_multiply`.

In [ ]:
def PYTORCH_multiply(x, y):
    """Hint: x * y  OR  torch.mul(x, y)"""
    return NotImplemented   # TODO

np.random.seed(0)
X = torch.from_numpy(np.random.randint(-1000, 1000, size=3000))
Y = torch.from_numpy(np.random.randint(-1000, 1000, size=3000))
print(PYTORCH_multiply(X, Y))

### 4.4 ReLU

$$\text{ReLU}(x) = \max(0, x)$$

This is the most common activation function in deep learning. It zeros out negative values.

**Your Task:** Implement `PYTORCH_relu`.

In [ ]:
def PYTORCH_relu(X):
    """
    Apply ReLU element-wise.
    Hint: torch.clamp(X, min=0)  OR  torch.relu(X)  OR  F.relu(X)
    """
    return NotImplemented   # TODO

X = torch.tensor([[-3.0, 0.0, 2.5],
                   [ 1.0, -1.0, 4.0]])
print(PYTORCH_relu(X))
# Expected:
# tensor([[0.0, 0.0, 2.5],
#         [1.0, 0.0, 4.0]])

---
## **5. Tensor Shape Manipulation**

Getting shapes right is one of the most common struggles in deep learning. Master these operations!

| Operation | What it does |
|---|---|
| `.reshape(new_shape)` | Rearrange elements into a new shape |
| `.transpose(dim0, dim1)` | Swap two dimensions |
| `.permute(dims)` | Reorder ALL dimensions at once |
| `.flatten()` | Collapse to 1D |
| `.unsqueeze(dim)` | Add a new dimension of size 1 |
| `.squeeze(dim)` | Remove dimensions of size 1 |
| `torch.cat([...], dim)` | Concatenate along existing dim |
| `torch.stack([...], dim)` | Stack creating a NEW dim |

In [ ]:
# reshape vs transpose — they look similar but are different!
X = torch.tensor([[1, 2, 3],
                   [4, 5, 6]])

print("Original shape:", X.shape)           # [2, 3]
print("reshape (3,2):\n", X.reshape(3, 2))  # rearranges elements
print("transpose:\n",     X.T)              # flips rows/cols — different result!

### 5.1 Flatten — Your Task

In [ ]:
def PYTORCH_flatten(input_tensor):
    """Flatten to 1D. Hint: torch.flatten(input_tensor)"""
    return NotImplemented   # TODO

np.random.seed(0)
X = torch.from_numpy(np.random.randint(-10, 10, size=(100, 100, 100)))
out = PYTORCH_flatten(X)
if out is not NotImplemented:
    print("Shape before:", X.shape)
    print("Shape after :", out.shape)   # Expected: torch.Size([1000000])

### 5.2 Unsqueeze — Your Task

Very common when you need to add a batch dimension: shape `(C, H, W)` → `(1, C, H, W)`.

In [ ]:
def PYTORCH_unsqueeze(X, dim):
    """Add a dimension at position `dim`. Hint: X.unsqueeze(dim) or torch.unsqueeze(X, dim)"""
    return NotImplemented   # TODO

X = torch.randn(3, 32, 32)  # C x H x W
result = PYTORCH_unsqueeze(X, 0)
if result is not NotImplemented:
    print("Before:", X.shape)       # [3, 32, 32]
    print("After :", result.shape)  # [1, 3, 32, 32] — batch dim added

### 5.3 Reshape — Your Task

In [ ]:
def PYTORCH_reshape(X, new_shape):
    """Reshape X to new_shape. Hint: X.reshape(new_shape) or torch.reshape(X, new_shape)"""
    return NotImplemented   # TODO

np.random.seed(0)
X = torch.from_numpy(np.random.randint(0, 10, size=(3, 10)))
print(PYTORCH_reshape(X, (2, 3, 5)))   # Expected shape: [2, 3, 5]

### 5.4 Transpose — Your Task

In [ ]:
def PYTORCH_transpose(X, dim0, dim1):
    """Swap two dimensions. Hint: X.transpose(dim0, dim1) or torch.transpose(X, dim0, dim1)"""
    return NotImplemented   # TODO

X = torch.randn(2, 3, 4)   # shape [2, 3, 4]
result = PYTORCH_transpose(X, 1, 2)
if result is not NotImplemented:
    print("Before:", X.shape)       # [2, 3, 4]
    print("After :", result.shape)  # [2, 4, 3]

### 5.5 Cat vs Stack — Know the Difference!

In [ ]:
A = torch.ones(2, 3)
B = torch.zeros(2, 3)

cat_result   = torch.cat([A, B], dim=0)    # merge along existing dim
stack_result = torch.stack([A, B], dim=0)  # create NEW dimension

print("A.shape:          ", A.shape)            # [2, 3]
print("cat  ([A,B], 0):  ", cat_result.shape)   # [4, 3]  ← same dims, doubled
print("stack([A,B], 0):  ", stack_result.shape) # [2, 2, 3] ← new first dim!

---
## **6. Autograd — Automatic Differentiation** ⭐ NEW

This is PyTorch's superpower. As you do forward computations, it builds a **computational graph**. Calling `.backward()` traverses the graph in reverse and computes all gradients via the chain rule — automatically.

### The key idea

When `requires_grad=True`, PyTorch records every operation on that tensor. After the forward pass, calling `loss.backward()` fills in the `.grad` attribute of all leaf tensors.

In [ ]:
# Simple example: y = x²,  dy/dx = 2x
x = torch.tensor(2.0, requires_grad=True)   # leaf node
y = x ** 2                                   # y = x²

print("y          :", y)            # tensor(4., grad_fn=PowBackward)
print("y.grad_fn  :", y.grad_fn)   # the backward function PyTorch created

y.backward()                        # compute dy/dx
print("x.grad     :", x.grad)      # tensor(4.)  ← dy/dx = 2*2 = 4 ✓

In [ ]:
# Peeking at the computational graph
a = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(4.0, requires_grad=True)
c = a * b + a   # c = ab + a

print("c.grad_fn            :", c.grad_fn)                    # AddBackward
print("c.grad_fn.next_fns   :", c.grad_fn.next_functions)     # MulBackward, AccumulateGrad

c.backward()
print("dc/da :", a.grad)   # dc/da = b + 1 = 5
print("dc/db :", b.grad)   # dc/db = a = 3

In [ ]:
# torch.no_grad() — use during validation / inference
# Disables graph building → faster + less memory

x = torch.randn(3, requires_grad=True)

with torch.no_grad():
    y = x * 2   # no graph built
    print("y.requires_grad inside no_grad:", y.requires_grad)   # False

print("y.requires_grad outside no_grad:", y.requires_grad)  # Still False

### ⚠️ Pitfall: Gradients Accumulate!

If you call `.backward()` multiple times without zeroing gradients first, they **add up** (this is by design for some advanced use cases, but almost never what you want in a standard training loop).

In [ ]:
x = torch.tensor(2.0, requires_grad=True)

for _ in range(3):
    y = x ** 2
    y.backward()
    print("x.grad:", x.grad)   # 4, 8, 12 — accumulating!

print()
# Fix: zero gradients before each backward
x.grad = None   # or in a training loop: optimizer.zero_grad()
for _ in range(3):
    y = x ** 2
    y.backward()
    print("x.grad (after zero):", x.grad)   # always 4
    x.grad.zero_()   # zero in-place

---
## **7. Building the Neural Network — nn.Module**

Every neural network in PyTorch is a Python class that:
1. Inherits from `nn.Module`
2. Defines layers in `__init__`
3. Defines the data flow in `forward()`

PyTorch handles all gradient tracking automatically for any `nn.Module`.

In [ ]:
class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()         # (B, 1, 28, 28) → (B, 784)
        self.fc1     = nn.Linear(28*28, 128) # 784 → 128
        self.fc2     = nn.Linear(128, 64)    # 128 → 64
        self.fc3     = nn.Linear(64, 10)     # 64  → 10 classes

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))   # apply activation AFTER linear
        x = F.relu(self.fc2(x))
        return self.fc3(x)        # raw logits — NO softmax here!

model = SimpleNN().to(device)
print(model)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")

---
## **8. model.train() vs model.eval()** ⭐ NEW

This is one of the most commonly forgotten steps. Certain layers — like **Dropout** and **BatchNorm** — behave differently during training vs inference.

| Call | Dropout | BatchNorm | When to use |
|---|---|---|---|
| `model.train()` | Active (random zeros) | Uses batch statistics | Training loop |
| `model.eval()` | Disabled | Uses running statistics | Validation / Test |

In [ ]:
# Simple demo with Dropout
dropout = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

dropout.train()   # training mode
print("train():", dropout(x))   # roughly half the values zeroed out

dropout.eval()    # eval mode
print("eval() :", dropout(x))   # all values pass through unchanged

In [ ]:
# ✅ Correct pattern to always follow:

# --- Training ---
model.train()                   # ← switch to training mode
for inputs, labels in []:       # placeholder — your real loader goes here
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()       # clear old gradients
    pred = model(inputs)        # forward pass
    loss = criterion(pred, labels)
    loss.backward()             # compute gradients
    optimizer.step()            # update weights

# --- Validation / Testing ---
model.eval()                    # ← switch to eval mode
with torch.no_grad():           # ← disable grad tracking
    for images, labels in []:   # placeholder
        images = images.to(device)
        outputs = model(images)
        predicted = outputs.argmax(dim=1)

---
## **9. Loss Functions**

Loss functions measure how wrong your model is. `loss.backward()` then computes how to make it less wrong.

| Loss | Use case | Notes |
|---|---|---|
| `nn.CrossEntropyLoss()` | Multi-class classification | Expects raw **logits** — do NOT apply softmax first! |
| `nn.MSELoss()` | Regression | Mean squared error |
| `nn.BCEWithLogitsLoss()` | Binary classification | More stable than BCE + Sigmoid |
| `nn.CTCLoss()` | Speech, OCR | No alignment needed |

In [ ]:
# CrossEntropyLoss example (most common for classification)
criterion = nn.CrossEntropyLoss()

# Fake batch: 4 samples, 3 classes
logits = torch.tensor([[2.0, 1.0, 0.1],
                        [0.5, 2.5, 0.3],
                        [0.1, 0.2, 3.0],
                        [1.5, 0.5, 0.2]])
labels = torch.tensor([0, 1, 2, 0])   # true class indices

loss = criterion(logits, labels)
print("Loss:", loss.item())

# ⚠️ CrossEntropyLoss applies softmax internally.
# Don't do: F.softmax(logits, dim=1) before passing to CELoss!
print("\nPredicted classes:", logits.argmax(dim=1))   # [0, 1, 2, 0] — correct!

---
## **10. Optimizers**

Optimizers update the model's weights using the computed gradients. The **learning rate** (`lr`) is the most important hyperparameter.

| Optimizer | When to use |
|---|---|
| `SGD(lr, momentum=0.9)` | Classic; good baseline with momentum |
| `Adam(lr=1e-3)` | Great default for most deep learning tasks |
| `AdamW(lr=1e-3)` | Adam with proper weight decay; preferred for Transformers |

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# The 3-line update rule — ORDER MATTERS:
# optimizer.zero_grad()   # ① clear old gradients
# loss.backward()         # ② compute new gradients
# optimizer.step()        # ③ update weights

print("Optimizer:", optimizer)

---
## **11. Putting It All Together — MNIST Classifier**

Let's train a fully connected network on MNIST from scratch, following the exact 5-step flow from the slides.

### 11.1 Data Preparation

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))   # normalize to [-1, 1]
])

trainset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
testset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True,  num_workers=2)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=32, shuffle=False, num_workers=2)

print(f"Training samples : {len(trainset)}")
print(f"Test samples     : {len(testset)}")

# Inspect one batch
images, labels = next(iter(trainloader))
print(f"Batch images shape: {images.shape}")   # [32, 1, 28, 28]
print(f"Batch labels shape: {labels.shape}")   # [32]

### 11.2 Model, Loss, and Optimizer

In [ ]:
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = SimpleNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Device:", device)
print(model)

### 11.3 Training Loop

In [ ]:
NUM_EPOCHS = 3
print('Starting Training')

for epoch in range(NUM_EPOCHS):
    model.train()                # ← always call before training
    running_loss = 0.0

    for i, (inputs, labels) in enumerate(trainloader, start=1):
        inputs, labels = inputs.to(device), labels.to(device)  # move to GPU

        optimizer.zero_grad()           # ① clear gradients
        pred = model(inputs)            # ② forward pass
        loss = criterion(pred, labels)  # ③ compute loss
        loss.backward()                 # ④ backward pass
        optimizer.step()               # ⑤ update weights

        running_loss += loss.item()
        if i % 300 == 0:
            print(f'[Epoch {epoch+1}, Batch {i:4d}] Loss: {running_loss/300:.4f}')
            running_loss = 0.0

print('Training Complete!')

### 11.4 Evaluation

In [ ]:
model.eval()                   # ← always call before evaluation
correct = 0
total   = 0

with torch.no_grad():          # ← disable gradient tracking
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)
        outputs   = model(images)
        predicted = outputs.argmax(dim=1)   # class with highest logit
        correct  += (predicted == labels).sum().item()
        total    += labels.numel()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')   # should be ~97%

---
## **12. Common Beginner Pitfalls Checklist** ⭐ NEW

Run through this mental checklist every time you write a training loop.

In [ ]:
# ─── Pitfall 1: Forgetting optimizer.zero_grad() ───────────────────────────
# Gradients accumulate across backward() calls.

# ❌ Wrong
# for inputs, labels in trainloader:
#     pred = model(inputs)
#     loss = criterion(pred, labels)
#     loss.backward()          # grads from THIS batch + ALL PREVIOUS batches!
#     optimizer.step()

# ✅ Correct
# for inputs, labels in trainloader:
#     optimizer.zero_grad()    # ← always first
#     pred = model(inputs)
#     loss = criterion(pred, labels)
#     loss.backward()
#     optimizer.step()

print("Pitfall 1: Always call optimizer.zero_grad() before loss.backward()")


# ─── Pitfall 2: Applying softmax before CrossEntropyLoss ───────────────────
# nn.CrossEntropyLoss already applies log-softmax internally.

logits = torch.tensor([[2.0, 1.0, 0.1]])   # raw model output
labels = torch.tensor([0])

criterion = nn.CrossEntropyLoss()

loss_correct = criterion(logits, labels)
loss_double  = criterion(F.softmax(logits, dim=1), labels)   # softmax applied twice!

print(f"\nPitfall 2: Correct loss={loss_correct.item():.4f}, Double-softmax loss={loss_double.item():.4f} ← wrong!")


# ─── Pitfall 3: Not switching modes ────────────────────────────────────────
net_with_dropout = nn.Sequential(nn.Linear(10, 10), nn.Dropout(0.9))
x = torch.ones(1, 10)

net_with_dropout.train()
out_train = net_with_dropout(x).sum().item()

net_with_dropout.eval()
out_eval = net_with_dropout(x).sum().item()

print(f"\nPitfall 3: train() output sum={out_train:.2f}, eval() output sum={out_eval:.2f}  ← different!")
print("  → Always call model.eval() before validation/testing!")

---
## **Further Reading**

- [PyTorch Beginner Basics](https://docs.pytorch.org/tutorials/beginner/basics/intro.html) — official step-by-step guide
- [view vs reshape vs transpose vs permute](https://jdhao.github.io/2019/07/10/pytorch_view_reshape_transpose_permute/) — essential shape-ops deep dive
- [Autograd: Automatic Differentiation](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html) — official autograd tutorial
- [torch.nn Reference](https://docs.pytorch.org/docs/stable/nn.html) — all layers, activations, losses
- [torch.optim Reference](https://docs.pytorch.org/docs/stable/optim.html) — all optimizers